# CSV Generator
Generates a CSV with: Number of Repetitions, Total Time, Mean Time Between Repetitions, Mean Force, and Mean Power.

> All processing functions are in `helpers.py`, which must be in the same directory as this notebook.

In [1]:
import os
import csv
import json
import matplotlib.pyplot as plt

from helpers import (
    prepare_accelerometer_data,
    detect_peaks,
    compute_step_metrics,
    compute_force_power,
)

In [2]:
# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────

data_path        = './data'
output_file_name = 'results.csv'
error_file_name  = 'errors.csv'
number_samples   = 208

header_file = (
    'Sample',
    'User ID',
    'Gender',
    'Age',
    'Number of Repetitions',
    'Total Time (s)',
    'Mean Time Between Repetitions (s)',
    'Mean Force (N)',
    'Mean Power (W)',
)

In [3]:
# ─────────────────────────────────────────────
# PLOT FUNCTION (lives here, not in helpers)
# ─────────────────────────────────────────────



In [4]:
# ─────────────────────────────────────────────
# CSV GENERATION
# ─────────────────────────────────────────────

with open(output_file_name, 'w', newline='', encoding='utf-8') as output_file, \
     open(error_file_name,  'w', newline='', encoding='utf-8') as error_file:

    writer = csv.writer(output_file)
    dump   = csv.writer(error_file)

    writer.writerow(header_file)
    dump.writerow(header_file)

    for sample in range(1, number_samples + 1):

        sample_path = data_path + '/' + str(sample)

        if not os.path.isdir(sample_path):
            print(f'Sample {sample}: directory not found — skipping.')
            continue

        file_list = os.listdir(sample_path)

        # ── Read JSON metadata ──────────────────────
        try:
            json_file = sample_path + '/' + [f for f in file_list if f.endswith('.json')][0]
            with open(json_file, 'r') as f:
                info_data = json.load(f)
        except (IndexError, FileNotFoundError, json.JSONDecodeError) as e:
            print(f'Sample {sample}: JSON error ({e}) — skipping.')
            continue

        record_row = (
            sample,
            info_data.get('user_id', '#N/A'),
            info_data.get('gender',  '#N/A'),
            info_data.get('age',     '#N/A'),
        )

        # ── Read accelerometer file ─────────────────
        try:
            accel_path = sample_path + '/' + [f for f in file_list if 'accelerometer' in f][0]
            with open(accel_path, 'r') as f:
                accel_raw = f.readlines()

            data_values = {'accelerometer': []}
            for line in accel_raw[1:]:
                parts = line.split()
                data_values['accelerometer'].append(
                    [eval(parts[0]), [eval(c) for c in parts[1:]]]
                )
        except (IndexError, FileNotFoundError) as e:
            print(f'Sample {sample}: accelerometer file error ({e}) — skipping.')
            dump.writerow(record_row + ('#N/A', '#N/A', '#N/A', '#N/A', '#N/A'))
            continue

        # ── Process & plot ──────────────────────────
        accel_data = prepare_accelerometer_data(data_values)
        plot_accelerometer(accel_data, peaks)
        peaks      = detect_peaks(accel_data)
        metrics    = compute_step_metrics(accel_data, peaks)

        if metrics['num_steps'] == 0 or metrics['mean_step_time'] == 0:
            print(f'Sample {sample}: no repetitions detected — written to errors.')
            dump.writerow(record_row + (0, metrics['total_time'], '#N/A', '#N/A', '#N/A'))
            continue

        # ── Compute Force & Power ───────────────────
        weight = info_data.get('weight_kg', None)
        if weight:
            mean_force, mean_power = compute_force_power(accel_data, weight)
        else:
            mean_force, mean_power = '#N/A', '#N/A'

        # ── Build and write row ─────────────────────
        record_row = record_row + (
            metrics['num_steps'],
            metrics['total_time'],
            metrics['mean_step_time'],
            mean_force,
            mean_power,
        )

        print(record_row)
        writer.writerow(record_row)

print('Done! Results saved to:', output_file_name)
print('Errors  saved to:',       error_file_name)

(1, 1, 'Female', 25, 21, 26.461, 1.191, 553.1202, 70961.3989)
(2, 2, 'Male', 27, 4, 29.131, 6.0813, 811.7685, 111866.9544)
(3, 3, 'Female', 25, 121, 30.883, 0.251, 543.4402, 81566.8696)
(4, 4, 'Female', 28, 6, 29.38, 4.6142, 551.1127, 78511.2889)
(5, 5, 'Female', 26, 20, 28.729, 1.4127, 550.7713, 76354.1782)
(6, 6, 'Female', 27, 17, 28.975, 1.6016, 532.7374, 74689.6795)
(7, 7, 'Male', 28, 14, 52.1, 3.3775, 856.0205, 214912.9112)
(8, 8, 'Female', 25, 161, 34.335, 0.2113, 542.4881, 90262.9323)
(9, 9, 'Female', 27, 46, 27.049, 0.5795, 533.813, 70033.875)
(10, 10, 'Male', 27, 7, 28.56, 4.3392, 749.5918, 102836.2143)
(11, 11, 'Male', 27, 154, 38.736, 0.2509, 717.4806, 133192.8064)
(12, 12, 'Female', 23, 29, 54.253, 1.8491, 5873.2856, 1528386.8496)
(13, 13, 'Female', 39, 19, 58.504, 2.7438, 541.3459, 153034.9106)
(14, 14, 'Female', 27, 33, 37.881, 1.1538, 533.4478, 98102.1139)
(15, 15, 'Female', 25, 7, 34.33, 3.4602, 570.4215, 94707.4261)
(16, 16, 'Female', 28, 3, 29.953, 7.7875, 557.5533, 7